# Assumption eval: Qwen3.5-4B, thinking on, Logic-Q near-miss set

Runs `Qwen/Qwen3.5-4B` in **thinking mode** on the dataset with vLLM, **one answer per prompt**. All prompts go to the engine at once, so it batches them continuously, and each answer is written to Drive **as soon as it finishes**. If Colab disconnects, re-run cells 2-5 and it resumes, skipping every prompt already answered.

**Settings (from the model card, thinking mode, general tasks):** temperature 1.0, top_p 0.95, top_k 20, min_p 0, presence_penalty 1.5, repetition_penalty 1.0, max 32,768 output tokens. Thinking is on by default (`enable_thinking=True`). There is no system prompt, and each user message is the dataset `prompt` exactly as it appears in the file.

**Before you start**
1. Runtime → Change runtime type → **A100 GPU** (L4 also works; the batch size adapts to the GPU memory).
2. Upload `logicq_data.jsonl` (and `logicq_data_pilot.jsonl` for a pilot) to `MyDrive/assumption_study/`.
3. Run cell 1 once. It restarts the runtime when it finishes. Then run cells 2 onward.

Results go to `MyDrive/assumption_study/results_<dataset name>.jsonl` (e.g. `results_logicq_data.jsonl`), one line per prompt. Only the **final answer** is saved (the thinking trace is discarded), because only the final answer is graded. The final answer is the text after the **last** `</think>`; `think_close_tags` records how many the output had (more than 1 means the model closed its thinking twice).

## 1. Install vLLM nightly (once per runtime; restarts the runtime)

In [ ]:
!pip install -q uv
!uv pip install --system -q vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly
!pip uninstall -y -q torchaudio
import vllm, torch; print("vllm", vllm.__version__, "| torch", torch.__version__)
import os; os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 118.0 MB/s eta 0:00:00


W1020 21:49:30.017000 5031 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1020 21:49:30.018000 5031 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1020 21:49:30.019000 5031 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


## 2. Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, hashlib
DATA_DIR    = "/content/drive/MyDrive/Fellowship/Fatima/assumption_eval_rerun"
DATASET     = "Data/logicq_data.jsonl"               

MODEL         = "Qwen/Qwen3.5-4B"
MAX_TOKENS    = 32768      
MAX_MODEL_LEN = 36864      
GPU_MEM_UTIL  = 0.92
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
MAX_NUM_SEQS = 256 if GPU_GB > 70 else 128 if GPU_GB > 35 else 64
print(f"GPU: {torch.cuda.get_device_name(0)}, {GPU_GB:.0f} GB -> MAX_NUM_SEQS = {MAX_NUM_SEQS}")

SAMPLING = dict(temperature=1.0, top_p=0.95, top_k=20, min_p=0.0,
                presence_penalty=1.5, repetition_penalty=1.0, max_tokens=MAX_TOKENS)

DATA_PATH = f"{DATA_DIR}/{DATASET}"
OUT_PATH  = f"{DATA_DIR}/results_{os.path.splitext(os.path.basename(DATASET))[0]}.jsonl"
assert os.path.exists(DATA_PATH), f"Upload {DATASET} to {DATA_DIR} first"
print("data   :", DATA_PATH)
print("results:", OUT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB, 42 GB -> MAX_NUM_SEQS = 128
data   : /content/drive/MyDrive/Fellowship/Fatima/assumption_eval_rerun/Data/logicq_data.jsonl
results: /content/drive/MyDrive/Fellowship/Fatima/assumption_eval_rerun/results_logicq_data.jsonl


## 3. Load the model (once per session)

In [ ]:
from vllm import LLM, SamplingParams
import vllm

common = dict(model=MODEL, dtype="bfloat16", max_model_len=MAX_MODEL_LEN,
              gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS, seed=0)
try:
    llm = LLM(**common, language_model_only=True)
except TypeError:
    llm = LLM(**common, limit_mm_per_prompt={"image": 0, "video": 0})
tok = llm.get_tokenizer()
print("loaded", MODEL, "with vllm", vllm.__version__)

W1020 21:58:10.486000 7267 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1020 21:58:10.488000 7267 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1020 21:58:10.489000 7267 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=7267) INFO 10-20 21:58:27 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 36864, 'max_num_seqs': 128, 'disable_log_stats': True, 'language_model_only': True, 'model': 'Qwen/Qwen3.5-4B'}


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

(MainProcess pid=7267) INFO 10-20 21:58:50 [model.py:733] Resolved architecture: Qwen3_5ForConditionalGeneration
(MainProcess pid=7267) INFO 10-20 21:58:50 [model.py:2156] Using max model len 36864


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

(MainProcess pid=7267) INFO 10-20 21:58:53 [model.py:951] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(MainProcess pid=7267) INFO 10-20 21:58:53 [model.py:1008] Disabled mm_prefix attention mode because multimodal inputs are configuration-disabled. Attention backends without mm_prefix support may now be selected.
(MainProcess pid=7267) INFO 10-20 21:58:54 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=7267) INFO 10-20 21:58:54 [config.py:759] Mamba cache mode is set to 'align' for Qwen3_5ForConditionalGeneration by default when prefix caching is enabled


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

(MainProcess pid=7267) INFO 10-20 21:58:54 [kernel.py:476] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=7267) WARNING 10-20 21:58:59 [system_utils.py:153] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=7267) INFO 10-20 22:04:25 [hf.py:647] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.
loaded Qwen/Qwen3.5-4B with vllm 0.32.1rc1.dev172+g9da5f2d31


## 4. Build requests (resume-aware)

In [ ]:
rows = [json.loads(l) for l in open(DATA_PATH)]

def render(prompt):
    assert isinstance(prompt, str), "prompt must be a string (the old trailing-comma bug made it a list)"
    return tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=True)

done = set()
if os.path.exists(OUT_PATH):
    for l in open(OUT_PATH):
        try:
            done.add(json.loads(l)["id"])
        except (json.JSONDecodeError, KeyError):
            pass   

requests = []
for row in rows:
    if row["id"] in done:
        continue
    text = render(row["prompt"])
    requests.append(dict(rid=row["id"], row=row, text=text,
                         n_prompt=len(tok(text, add_special_tokens=False)["input_ids"]),
                         seed=int(hashlib.sha256(row["id"].encode()).hexdigest()[:8], 16)))

requests.sort(key=lambda r: (-r["row"]["depth"], r["row"]["id"]))
print(f"{len(rows)} prompts; already answered: {len(done)}; to run now: {len(requests)}")
if requests:
    print("exact prompt tokens (Qwen tokenizer): max", max(r["n_prompt"] for r in requests),
          "| mean", round(sum(r["n_prompt"] for r in requests) / len(requests)))
    print("\n--- what the model receives (first request, last 400 chars) ---\n" + requests[0]["text"][-400:])

120 prompts; already answered: 0; to run now: 120
exact prompt tokens (Qwen tokenizer): max 1595 | mean 1153

--- what the model receives (first request, last 400 chars) ---
-legged and ink-stained is also thick-browed.

Here is what we know about Gideon:
Gideon is inventive.
Gideon is wax-stained.
Gideon is long-legged.
Gideon is flour-dusted.
Gideon is sunburned.
Gideon is ink-stained.
Gideon is barefoot.
Gideon is paint-splattered.
Gideon is grease-stained.
Gideon is observant.
Gideon is letter-writing.

Is Gideon sarcastic?<|im_end|>
<|im_start|>assistant
<think>



## 5. Run (continuous batching, every answer saved as it finishes)

In [ ]:
from collections import defaultdict

def final_answer(text):
    return text.rpartition("</think>")[2].strip() if "</think>" in text else ""

def record(req, out, t_run0):
    o = out.outputs[0]
    answer = final_answer(o.text)
    row = req["row"]
    return {
        "id": row["id"], "puzzle_id": row["puzzle_id"], "variant": row["variant"],
        "twin_type": row["twin_type"], "correct_answer": row["correct_answer"],
        "missing_fact": row["missing_fact"], "goal": row["goal"],
        "depth": row["depth"], "band": row["band"],
        "answer": answer,
        "think_close_tags": o.text.count("</think>"),   # more than 1: the answer was cut after the last one
        "finish_reason": o.finish_reason, "cut_off": o.finish_reason == "length",
        "has_final_answer": bool(answer),
        "prompt_tokens": len(out.prompt_token_ids or []) or req["n_prompt"],
        "output_tokens": len(o.token_ids),
        "elapsed_s_since_run_start": round(time.time() - t_run0, 1),
        "prompt": row["prompt"],
    }

def save(rec, f):
    f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush(); os.fsync(f.fileno())

def make_params(req):
    return SamplingParams(**SAMPLING, seed=req["seed"])

by_rid = {r["rid"]: r for r in requests}
stats = defaultdict(lambda: {"n": 0, "tokens": 0, "cut": 0})
t0 = time.time(); n_done = 0; total_tokens = 0

def report(rec):
    global n_done, total_tokens
    n_done += 1; total_tokens += rec["output_tokens"]
    s = stats[rec["band"]]; s["n"] += 1; s["tokens"] += rec["output_tokens"]; s["cut"] += rec["cut_off"]
    el = time.time() - t0
    print(f'[{n_done}/{len(requests)}] {rec["id"]}: {rec["output_tokens"]} tok, '
          f'{rec["finish_reason"]}, final answer: {rec["has_final_answer"]} | '
          f'{el/60:.1f} min, {total_tokens/el:.0f} tok/s overall', flush=True)

if not requests:
    print("Nothing to run: every prompt is already answered.")
else:
    with open(OUT_PATH, "a") as f:
        try:
            eng = llm.llm_engine
            for r in requests:
                eng.add_request(r["rid"], {"prompt": r["text"]}, make_params(r))
            while eng.has_unfinished_requests():
                for out in eng.step():
                    if out.finished:
                        rec = record(by_rid[out.request_id], out, t0); save(rec, f); report(rec)
        except (AttributeError, TypeError) as e:
            print("engine API not available (", e, ") -> chunked generate()")
            saved = {json.loads(l)["id"] for l in open(OUT_PATH) if l.strip()}
            left = [r for r in requests if r["rid"] not in saved]
            CHUNK = MAX_NUM_SEQS
            for i in range(0, len(left), CHUNK):
                part = left[i:i + CHUNK]
                outs = llm.generate([{"prompt": r["text"]} for r in part], [make_params(r) for r in part], use_tqdm=True)
                for r, out in zip(part, outs):
                    rec = record(r, out, t0); save(rec, f); report(rec)
    el = time.time() - t0
    print(f"\nDone: {n_done} answers in {el/60:.1f} min, {total_tokens} output tokens, {total_tokens/el:.0f} tok/s (batched)")
    for b, s in sorted(stats.items()):
        print(f'  band {b}: {s["n"]} answers, mean {s["tokens"]/s["n"]:.0f} tokens, cut off {s["cut"]}')

(MainProcess pid=7267) WARNING 10-20 22:20:58 [input_processor.py:408] Passing raw prompts to InputProcessor is deprecated and will be removed in the future. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().
[1/120] nm-d07-03-twin_yes: 4355 tok, stop, final answer: True | 2.6 min, 28 tok/s overall
[2/120] nm-d11-01-twin_yes: 5492 tok, stop, final answer: True | 3.3 min, 49 tok/s overall
[3/120] nm-d10-07-twin_yes: 5822 tok, stop, final answer: True | 3.6 min, 73 tok/s overall
[4/120] nm-d08-05-twin_yes: 6198 tok, stop, final answer: True | 3.9 min, 94 tok/s overall
[5/120] nm-d10-05-twin_yes: 6258 tok, stop, final answer: True | 3.9 min, 120 tok/s overall
[6/120] nm-d09-05-twin_yes: 6795 tok, stop, final answer: True | 4.3 min, 136 tok/s overall
[7/120] nm-d09-01-twin_yes: 6830 tok, stop, final answer: True | 4.3 min, 161 tok/s overall
[8/120] nm-d11-08-twin_no: 6896 tok, stop, final answer: True | 4.3 min, 187 tok/s overall
[9/120] nm-d12-05-twin

## 6. Summary of everything saved so far

In [ ]:
import pandas as pd
res = pd.DataFrame([json.loads(l) for l in open(OUT_PATH) if l.strip()])
print(len(res), "prompts answered")
display(res.groupby(["band", "variant"]).agg(prompts=("id", "size"),
        mean_tokens=("output_tokens", "mean"), max_tokens=("output_tokens", "max"),
        cut_off=("cut_off", "sum"), no_final_answer=("has_final_answer", lambda x: (~x).sum())).round(0))

if "total_tokens" in globals() and total_tokens and (time.time() - t0) > 0:
    tps = total_tokens / (time.time() - t0)
    hrs = 120 * res["output_tokens"].mean() / tps / 3600
    print(f"full set: ~{hrs:.1f} h at {tps:.0f} tok/s (+25% margin: {hrs*1.25:.1f} h). "
          "Throughput rises with more prompts in flight, so a small pilot's rate is pessimistic.")

120 prompts answered


prompts  mean_tokens  max_tokens  cut_off  \
band  variant                                                     
11-12 twin_no              10       9214.0       11392        0   
      twin_yes             10       8332.0       11553        0   
      underspecified       20      17524.0       32768        1   
7-8   twin_no              10       8579.0       10161        0   
      twin_yes             10       6594.0        7201        0   
      underspecified       20      14349.0       17868        0   
9-10  twin_no              10       9228.0       11905        0   
      twin_yes             10       8587.0       21457        0   
      underspecified       20      15313.0       21063        0   

                      no_final_answer  
band  variant                          
11-12 twin_no                       0  
      twin_yes                      0  
      underspecified                0  
7-8   twin_no                       0  
      twin_yes                      0  
      underspecified                0  
9-10  twin_no                       0  
      twin_yes                      0  
      underspecified                0

full set: ~0.3 h at 1190 tok/s (+25% margin: 0.4 h). Throughput rises with more prompts in flight, so a small pilot's rate is pessimistic.


## 7. Read one answer

In [9]:
r = res.iloc[0]
print(r["id"], "| correct:", r["correct_answer"], "| missing fact:", r["missing_fact"])
print(f'{r["output_tokens"]} tokens, finish: {r["finish_reason"]}\n')
print("FINAL ANSWER:\n" + (r["answer"] or "(none: cut off)"))

nm-d07-03-twin_yes | correct: yes | missing fact: gum-chewing
4355 tokens, finish: stop

FINAL ANSWER:
Based on the rules and facts provided, we can deduce whether Rafael is yo-yo-spinning step-by-step.

**1. Determine if Rafael is Ticklish**
*   **Rule:** Rafael is ticklish if Rafael is curly-haired, barefoot, and letter-writing.
*   **Facts:** Rafael is curly-haired, barefoot, and letter-writing.
*   **Conclusion:** Rafael is ticklish.

**2. Determine if Rafael is Adventurous**
*   **Rule:** Once Rafael is tenured, tent-pitching, and ticklish, Rafael is adventurous as well.
*   **Facts/Rules:** Rafael is tenured, tent-pitching (given facts), and ticklish (derived above).
*   **Conclusion:** Rafael is adventurous.

**3. Determine if Rafael is Widowed**
*   **Rule:** Whenever Rafael is sunburned and adventurous, Rafael is also widowed.
*   **Facts:** Rafael is sunburned (given fact) and adventurous (derived above).
*   **Conclusion:** Rafael is widowed.

**4. Determine if Rafael is Spo